# Silver — GDP Indicators
Cleans and enriches Bronze GDP data: adds a proper date column for each quarter and rounds values.

**Source:** `bronze.statistics_iceland_raw`  
**Output:** `silver.gdp_indicators` (Delta table)  
**Date rule:** Q1 → Jan 1, Q2 → Apr 1, Q3 → Jul 1, Q4 → Oct 1

In [ ]:
df_raw = spark.read.table("bronze.statistics_iceland_raw").filter("dataset = 'gdp'")
df_raw.createOrReplaceTempView("v_gdp_raw")

In [ ]:
df_silver = spark.sql("""
    WITH transformed AS (
        SELECT
            period_raw AS quarter,
            CAST(SUBSTRING(period_raw, 1, 4) AS INT) AS year,
            CAST(SUBSTRING(period_raw, 6, 1) AS INT) AS q,
            CAST(value_raw AS DOUBLE) AS value,
            TO_DATE(CONCAT(SUBSTRING(period_raw, 1, 4), '-', 
                CASE SUBSTRING(period_raw, 6, 1)
                    WHEN '1' THEN '01' WHEN '2' THEN '04'
                    WHEN '3' THEN '07' WHEN '4' THEN '10'
                END, '-01')) AS date,
            ingested_at
        FROM v_gdp_raw
        WHERE value_raw NOT IN ('.', '') AND value_raw IS NOT NULL
    ),
    latest_records AS (
        SELECT 
            quarter, 
            year, 
            q, 
            value, 
            date, 
            ingested_at,
            ROW_NUMBER() OVER (PARTITION BY date ORDER BY ingested_at DESC) AS rn
        FROM transformed
    )
    SELECT 
        quarter, 
        year, 
        q, 
        date, 
        'gdp_yoy_growth' AS metric, 
        value, 
        CURRENT_TIMESTAMP() AS processed_at
    FROM latest_records
    WHERE rn = 1
""")

df_silver.createOrReplaceTempView("v_gdp_silver_staged")

if df_silver.isEmpty():
    print("No valid GDP data found after cleaning. Exiting.")
    mssparkutils.notebook.exit("no-data")

In [ ]:
if not spark.catalog.tableExists("silver.gdp_indicators"):
    df_silver.write.format("delta").saveAsTable("silver.gdp_indicators")
    print("Created silver.gdp_indicators.")
else:
    spark.sql("""
        MERGE INTO silver.gdp_indicators AS target
        USING v_gdp_silver_staged AS source
        ON target.year = source.year 
        AND target.q = source.q 
        AND target.metric = source.metric
        WHEN MATCHED THEN UPDATE SET
            target.quarter      = source.quarter,
            target.value        = source.value,
            target.date         = source.date,
            target.processed_at = source.processed_at
        WHEN NOT MATCHED THEN INSERT (quarter, year, q, date, metric, value, processed_at)
        VALUES (source.quarter, source.year, source.q, source.date, source.metric, source.value, source.processed_at)
    """)

print(f"Merge complete. Final count: {spark.table('silver.gdp_indicators').count()}")